<a href="https://colab.research.google.com/github/financieras/articulos/blob/main/Redes_Neuronales_desde_cero.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Redes Neuronales desde cero: Comprendiendo el Cerebro Artificial**
## **Una guía práctica para implementar un perceptrón multicapa, paso a paso con el dataset Iris**

Las redes neuronales son el corazón de la inteligencia artificial moderna. Reconocen rostros en tus fotos, traducen idiomas en tiempo real, y impulsarán los próximos avances en medicina, ciencia y tecnología. Detrás de esta aparente magia hay matemáticas elegantes: multiplicaciones matriciales, derivadas y un algoritmo llamado backpropagation que permite a la red «aprender» de sus errores.

En este artículo construiremos una red neuronal desde cero usando únicamente NumPy. Sin TensorFlow, sin PyTorch, sin abstracciones ocultas. Usaremos el dataset Iris como ejemplo práctico: clasificar flores en tres especies basándonos en cuatro mediciones físicas. Implementaremos una arquitectura con dos capas ocultas para que puedas ver exactamente cómo fluyen los datos y cómo se propagan los errores.

Si sabes Python y recuerdas cómo derivar una función, tienes todo lo necesario. Al terminar, entenderás cada operación matemática que ocurre cuando una red neuronal aprende, y además compararemos nuestros resultados con la implementación de Scikit-learn.

## Tabla de Contenido

1. **Introducción: ¿Qué es una red neuronal?**
   - La analogía del estudiante
   - Por qué importa entender los fundamentos

2. **El Dataset Iris: clasificando flores**
   - Descripción del problema
   - Preparación de los datos

3. **La neurona: la unidad fundamental**
   - Weighted sum y función de activación
   - Notación vectorial simplificada

4. **Arquitectura de dos capas ocultas: 4→16→8→3**
   - Por qué dos capas ocultas
   - Dimensiones de cada capa explicadas

5. **Forward propagation: el flujo hacia adelante**
   - Cómo los datos viajan desde la entrada hasta la salida
   - Código paso a paso

6. **Midiendo el error: la función de pérdida**
   - Cross-Entropy para clasificación multiclase
   - Intuición detrás de la fórmula

7. **Backward propagation: el alma del aprendizaje**
   - La regla de la cadena explicada suavemente
   - Un ejemplo numérico completo con dos capas

8. **Implementación completa en NumPy**
   - Todo el código junto
   - El bucle de entrenamiento

9. **Entrenamiento y resultados**
   - Visualizando el aprendizaje
   - Curvas de pérdida y precisión

10. **Comparación con Scikit-learn**
    - MLPClassifier en acción
    - Análisis de diferencias

11. **Conclusión y siguientes pasos**
    - Qué hemos aprendido
    - Dónde ir desde aquí

# 1. Introducción: ¿Qué es una red neuronal?

Imagina que eres un becario que afronta sus primeas prácticas de biología. Comienzas en un laboratorio de botánica aprendiendo a clasificar las diferentes variedades de la flor Iris. Tu jefe te coloca frente a tres montones de flores —setosa, versicolor y virginica— y te da una regla. «Mide el sépalo y el pétalo de cada flor», te dice, «y clasifícalas correctamente». Tras semanas de práctica, desarrollas un sexto sentido: cuando ves una flor con pétalo largo y estrecho, sabes que es virginica; cuando el pétalo es corto y ancho, probablemente es setosa. Has aprendido a clasificar.

Las redes neuronales artificiales funcionan de manera similar, solo que en lugar de un cerebro biológico usan matemáticas, y en lugar de semanas de práctica usan iteraciones algorítmicas. Una red neuronal es, en esencia, un sistema que aprende a transformar entradas (mediciones de una flor) en salidas correctas (la especie correspondiente) a través de un proceso de ajuste iterativo de parámetros internos. Estos parámetros internos se llaman pesos, y el proceso de ajustarlos es lo que denominamos aprendizaje.

### La analogía del estudiante

Para entender cómo aprende una red neuronal, piensa en un estudiante preparándose para un examen. El proceso tiene cuatro etapas claramente definidas que se repiten ciclo tras ciclo. Primero, el estudiante responde las preguntas del examen utilizando todo lo que sabe hasta el momento —esto es la propagación hacia adelante o forward propagation, donde los datos fluyen a través de la red produciendo una predicción. Segundo, el profesor califica el examen y determina cuántas preguntas falló y en cuáles —esto es el cálculo de la función de pérdida, que mide el error de la predicción. Tercero, el estudiante revisa sus respuestas, identifica dónde se equivocó y entiende qué conceptos tenía mal —esto es la propagación hacia atrás o backward propagation, donde el error se distribuye hacia todos los pesos que contribuyeron a él. Cuarto, el estudiante ajusta su comprensión: si un concepto específico le generó confusión, prestará más atención a ese tema la próxima vez —esto es la actualización de pesos, donde cada parámetro se modifica proporcionalmente a su contribución al error.

Repite este ciclo suficientes veces —cientos o miles de veces— y el estudiante dominará la materia. De manera análoga, la red neuronal repetirá el ciclo de predicción, cálculo de error, distribución del error y actualización de pesos hasta que sus predicciones sean suficientemente precisas. Lo notable es que el algoritmo que gobierna este proceso, la propagación hacia atrás, es fundamentalmente el mismo que usamos para calcular derivadas en cálculo: la regla de la cadena. Nada más intimidante que derivadas de toda la vida aplicadas sistemáticamente.

### ¿Por qué importa entender los fundamentos?

En la era de TensorFlow, PyTorch y bibliotecas de alto nivel, podrías entrenar una red neuronal con cinco líneas de código sin escribir una sola ecuación. ¿Para qué molestarse entonces en entender cómo funciona internamente? La respuesta es que cuando tu modelo no converge, cuando sobreajusta dramáticamente, cuando produce resultados absurdos, necesitas diagnosticar el problema, y para eso necesitas saber qué está pasando bajo el capó. Los ingenieros de machine learning que construyen sistemas de producción no son aquellos que memorizan APIs, sino aquellos que comprenden los fundamentos lo suficientemente bien para diseñar arquitecturas personalizadas, diagnosticar problemas de convergencia y optimizar el rendimiento de sus modelos.

Este artículo te lleva detrás de las escenas. Implementaremos una red neuronal completa desde cero usando únicamente NumPy, sin abstracciones ocultas, viendo cada operación matemática. Usaremos el dataset Iris como ejemplo práctico: 150 flores con cuatro mediciones cada una, clasificadas en tres especies. La arquitectura tendrá dos capas ocultas —16 neuronas en la primera, 8 en la segunda— para que puedas ver exactamente cómo fluyen los datos a través de múltiples capas y cómo los errores se propagan hacia atrás a través de cada una. Al final, compararemos nuestra implementación nativa con la de Scikit-learn para validar que todo funciona correctamente y entender qué ganan las bibliotecas profesionales en términos de optimización.

Si sabes programar en Python y recuerdas lo básico de derivadas, tienes todo lo necesario. Comencemos.

# 2. El Dataset Iris: clasificando flores

El dataset Iris es el «Hola Mundo» del aprendizaje automático. Creado por el estadístico Ronald Fisher en 1936, contiene 150 flores de tres especies diferentes —setosa, versicolor y virginica— con cuatro mediciones morfológicas cada una. A pesar de su antigüedad, sigue siendo el ejemplo perfecto para aprender redes neuronales porque equilibra simplicidad con suficiente complejidad para demostrar conceptos importantes.

### Descripción del problema

Nuestro objetivo es construir un clasificador que, dada una flor con sus cuatro mediciones, prediga correctamente su especie. Las cuatro características disponibles son: longitud del sépalo, ancho del sépalo, longitud del pétalo y ancho del pétalo, todas expresadas en centímetros. Las tres clases son mutuamente excluyentes: cada flor pertenece a exactamente una especie. Esto es un problema de clasificación multiclase con tres categorías, y la red neuronal debe outputear tres probabilidades —una por cada clase— que sumen a uno.

La gracia del dataset Iris es que las clases no son perfectamente separables linealmente. Setosa está claramente separada de las otras dos, pero versicolor y virginica se solapan parcialmente en el espacio de características. Una regresión logística simple (una sola neurona) tendría dificultades aquí, pero una red con capas ocultas puede aprender fronteras de decisión no lineales que separen mejor las clases. Por eso usaremos dos capas ocultas: para que la red pueda representar patrones más complejos que una sola transformación lineal.

Visualizar los datos ayuda a intuición. Si graficamos solo dos características —digamos, longitud del pétalo versus ancho del pétalo— podemos ver que setosa forma un grupo compacto en la zona de pétalos pequeños, mientras que versicolor y virginica se mezclan en la zona de pétalos más grandes pero distintos. Tu cerebro humano reconoce estos patrones inmediatamente, y una red neuronal bien entrenada también los capturará, solo que usando matemáticas en lugar de neuronas biológicas.

### Preparación de los datos

Antes de alimentar los datos a la red, necesitamos realizar tres pasos de preprocesamiento que son universales en machine learning: codificación de etiquetas, división en conjuntos de entrenamiento y prueba, y normalización de características.

Las etiquetas originales vienen como números enteros —0 para setosa, 1 para versicolor, 2 para virginica— pero para entrenar una red neuronal necesitamos codificación one-hot. Esto significa representar cada etiqueta como un vector con un 1 en la posición de la clase correcta y 0 en las demás. Así, setosa se convierte en [1, 0, 0], versicolor en [0, 1, 0] y virginica en [0, 0, 1]. Esta representación permite que la red outputee tres probabilidades que comparamos directamente con el vector objetivo.

Dividimos los datos en dos conjuntos: entrenamiento (80%) y prueba (20%). El conjunto de entrenamiento se usa para ajustar los pesos de la red, mientras que el conjunto de prueba permanece invisible durante el entrenamiento y sirve para evaluar qué tan bien generaliza el modelo a flores que no ha visto nunca. Esta separación es crucial para detectar sobreajuste, que ocurre cuando la red memoriza los ejemplos de entrenamiento en lugar de aprender patrones generalesizables.

La normalización es el paso más importante desde la perspectiva numérica. Las cuatro características del Iris tienen escalas muy diferentes —la longitud del sépalo va de 4 a 8 centímetros, mientras que el ancho del pétalo va de 0.1 a 2.5 centímetros— y si alimentamos estos valores directamente a la red, las características con mayor magnitud dominarían el aprendizaje. Estandarizamos cada característica restando su media y dividiendo por su desviación estándar, transformándolas para que tengan media cero y desviación estándar uno. Esto mantiene la información relativa de cada característica mientras equilibra su influencia en el aprendizaje.

In [1]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Cargar el dataset
iris = load_iris()
X = iris.data  # 150 flores × 4 características
y = iris.target  # 150 etiquetas (0, 1, o 2)

# División train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Normalización
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"Entrenamiento: {X_train.shape[0]} flores")
print(f"Prueba: {X_test.shape[0]} flores")

Entrenamiento: 120 flores
Prueba: 30 flores


Con los datos preparados, estamos listos para definir la arquitectura de nuestra red. En la siguiente sección explicaremos cómo organizamos las neuronas en capas y por qué elegimos la estructura específica 4→16→8→3.

# 3. La neurona: la unidad fundamental

La neurona artificial es la unidad básica de computación en una red neuronal. Su funcionamiento es notablemente simple, casi decepcionante si esperabas complejidad: recibe números como entrada, los multiplica por otros números (los pesos), suma todo, y opcionalmente transforma el resultado con una función matemática. De esta simplicidad emerge toda la capacidad de aprendizaje de las redes neuronales, siempre que tengamos suficientes neuronas conectadas apropiadamente.

### Suma ponderada: la combinación lineal

Imagina una neurona recebendo información de varias fuentes. Cada fuente tiene una importancia diferente, expresada como un peso. Una flor con pétalo muy largo probablemente es virginica, así que el peso asociado a la longitud del pétalo debería ser alto para esa clase. El sépalo quizás importa menos, así que su peso sería menor. La neurona combina todas estas entradas multiplicando cada una por su peso correspondiente y sumando el resultado.

Matemáticamente, para una neurona con entradas $x_1, x_2, \ldots, x_n$ y pesos correspondientes $w_1, w_2, \ldots, w_n$, más un sesgo $b$, calculamos la suma ponderada como：

$$z = w_1x_1 + w_2x_2 + \ldots + w_nx_n + b$$

El sesgo (bias en inglés) es como un término independiente en una línea: permite desplazar la activación de la neurona independientemente de las entradas. Si todos los pesos fueran cero pero el sesgo fuera positivo, la neurona aún podría activarse. Esta flexibilidad es crucial para que la red aprenda correctamente.

En notación vectorial, que es cómo realmente implementamos esto en código, si $\mathbf{x}$ es un vector columna con las entradas y $\mathbf{w}$ es un vector columna con los pesos, entonces el producto punto $\mathbf{w}^T\mathbf{x}$ es exactamente la suma ponderada, y podemos escribir $z = \mathbf{w}^T\mathbf{x} + b$. Esta notación compactará enormemente nuestro código cuando implementemos capas enteras de neuronas.

### Función de activación: la no linealidad

La suma ponderada z es un número real que puede tomar cualquier valor entre menos infinito y más infinito. Pero queremos que la neurona outputee algo útil: para clasificación, probabilidades entre cero y uno. Aquí entra la función de activación, que transforma z en un valor con las propiedades que necesitamos.

Hay varias funciones de activación, cada una con sus características. La función sigmoid transforma $z$ en un valor entre cero y uno, lo cual la hace ideal para probabilidades. Sin embargo, tiene un problema llamado gradientes vanishing: cuando $z$ es muy grande positivo o muy grande negativo, la derivada de sigmoid es casi cero, lo cual dificulta el aprendizaje en redes profundas. La función ReLU (Rectified Linear Unit) resuelve este problema con una definición simple：

$$\text{ReLU}(z) = \max(0, z)$$

Si $z$ es positivo, lo deja pasar sin modificarlo; si es negativo, lo convierte en cero. Es computacionalmente eficiente y mitiga el problema de gradientes vanishing, por lo que es la elección estándar para capas ocultas.

La función Softmax es diferente: se aplica a un vector completo de neuronas para convertirlo en probabilidades que suman a uno. Si la capa de salida tiene tres neuronas (una por cada clase de flor), Softmax transforma sus tres valores $z$ en tres probabilidades que indican qué tan probable es cada clase. La probabilidad de la clase $i$ es：

$$\text{Softmax}(z_i) = \frac{e^{z_i}}{\sum_{j=1}^{K} e^{z_j}}$$

donde $K$ es el número de clases. Esta exponenciación amplifica las diferencias: si una neurona tiene un $z$ ligeramente mayor que las otras, su probabilidad será sustancialmente mayor.

### Una neurona en acción

Veamos cómo se ve esto en código para una sola neurona. Aunque en la práctica trabajaremos con capas enteras, entender la neurona individual es el fundamento.

In [2]:
import numpy as np

# Una flor con 4 características (ya normalizadas)
x = np.array([0.5, -0.2, 1.1, -0.3])  # longitud sépalo, ancho sépalo, etc.

# Pesos para una neurona de la capa oculta
w = np.array([0.8, -0.5, 1.2, 0.3])
b = 0.1

# Suma ponderada
z = np.dot(w, x) + b  # o equivalentemente: w @ x + b

# ReLU para capa oculta
a = np.maximum(0, z)

print(f"Pesos: {w}")
print(f"Entrada: {x}")
print(f"Suma ponderada z = {z:.4f}")
print(f"Activación ReLU a = {a:.4f}")

Pesos: [ 0.8 -0.5  1.2  0.3]
Entrada: [ 0.5 -0.2  1.1 -0.3]
Suma ponderada z = 1.8300
Activación ReLU a = 1.8300


Este código implementa exactamente una neurona: recibe cuatro entradas, las combina con cuatro pesos más un sesgo, y aplica ReLU. En una capa oculta real, tendríamos 16 neuronas como esta, cada una con sus propios 4 pesos y su propio sesgo. En lugar de escribir un bucle para cada neurona, usamos multiplicación de matrices: una matriz de pesos de dimensiones $4 \times 16$ multiplicada por el vector de entrada de dimensiones $4$ produce un vector de $16$ activaciones de una sola vez.

La belleza de las redes neuronales está en que esta simplicidad se composiciona: neuronas individuales que hacen operaciones triviales, conectadas en capas, producen colectivamente comportamiento complejo e interesante. Con suficientes neuronas y las conexiones correctas, la red puede aproximar cualquier función continua, un resultado matemático profundo llamado teorema de aproximación universal. Pero de nada sirve tener muchas neuronas si no sabemos cómo ajustar sus pesos para que aprendan. Eso es precisamente lo que hace la propagación hacia atrás, el tema que abordaremos en detalle más adelante. Primero, veamos cómo organizamos las neuronas en capas.

# 4. Arquitectura de dos capas ocultas: 4→16→8→3

Una neurona sola, por más interesante que sea su funcionamiento, tiene capacidades limitadas. Puede aprender fronteras de decisión lineales, separando el espacio con un hiperplano, pero no puede capturar patrones más complejos como curvas o interacciones entre características. Para eso necesitamos muchas neuronas trabajando juntas, organizadas en capas que se comunican entre sí. Una red neuronal con múltiples capas ocultas puede representar funciones arbitrariamente complejas, y esa capacidad es precisamente lo que hace useful a las redes neuronales profundas.

### ¿Por qué dos capas ocultas?

La respuesta corta es que dos capas ocultas son suficientes para el dataset Iris y nos permiten ver cómo funciona la propagación a través de múltiples etapas sin la complejidad de arquitecturas más profundas. La respuesta más interesante es que cada capa adicional permite representar combinaciones de características cada vez más abstractas.

En la primera capa oculta, las neuronas probablemente aprenderán a detectar patrones simples: una neurona podría activarse cuando la longitud del pétalo es grande, otra cuando el ancho del sépalo es pequeño, y así sucesivamente. Estas son características de bajo nivel directamente derivadas de las mediciones originales. La segunda capa oculta toma estas características simples y las combina: una neurona podría activarse cuando simultáneamente tenemos «pétalo largo Y sépalo ancho», detectando patrones más elaborados que involucran múltiples características simultáneamente. La capa de salida entonces combina estas características de segundo nivel para producir las probabilidades finales de cada clase.

Este proceso de construcción jerárquica de representaciones es exactamente lo que hace poderosas a las redes neuronales profundas. Cada capa transforma la representación anterior en algo un poco más abstracto y útil para la tarea final. Para un problema simple como Iris, dos capas son más que suficiente; para reconocimiento de imágenes, necesitaríamos cientos de capas donde las primeras detectan bordes y texturas, las intermedias detectan formas, y las finales reconocen objetos completos.

### La arquitectura 4→16→8→3 explicada

Nuestra red tiene cuatro capas en total: la capa de entrada, dos capas ocultas, y la capa de salida. Los números describen cuántas neuronas tiene cada capa, y la flecha indica la dirección del flujo de datos.

La capa de entrada tiene 4 neuronas, una por cada característica del dataset Iris. No hay procesamiento aquí: estas neuronas simplemente reciben los valores normalizados de longitud del sépalo, ancho del sépalo, longitud del pétalo y ancho del pétalo. Si nuestro input es una flor, la capa de entrada es simplemente sus cuatro mediciones en forma de vector.

La primera capa oculta tiene 16 neuronas. Cada una de estas 16 neuronas recibe las 4 entradas, aplica sus propios pesos, sesgo y función de activación, y produce un salida. Matemáticamente, esto significa que necesitamos una matriz de pesos de dimensiones 4×16 (cada una de las 16 neuronas tiene 4 pesos) más un vector de sesgos de 16 elementos. El resultado es un vector de 16 activaciones que pasa a la siguiente capa.

La segunda capa oculta tiene 8 neuronas. Cada una de estas 8 neuronas recibe las 16 salidas de la capa anterior como entradas. La matriz de pesos aquí tiene dimensiones 16×8, más un vector de sesgos de 8 elementos. Esta segunda transformación refina las características que ya combinó la primera capa, creando representaciones de nivel superior.

La capa de salida tiene 3 neuronas, una por cada clase de flor. Estas neuronas reciben las 8 activaciones de la segunda capa oculta y producen un vector de 3 valores que, después de Softmax, se convierten en probabilidades para setosa, versicolor y virginica. La matriz de pesos final es de 8×3, con un vector de sesgos de 3 elementos.

### Contando los parámetros

Para apreciar la escala de la red, contemos cuántos parámetros totales tenemos que aprender. La primera capa tiene 4×16 pesos más 16 sesgos = 64 + 16 = 80 parámetros. La segunda capa tiene 16×8 pesos más 8 sesgos = 128 + 8 = 136 parámetros. La capa de salida tiene 8×3 pesos más 3 sesgos = 24 + 3 = 27 parámetros. En total, nuestra red tiene 80 + 136 + 27 = 243 parámetros ajustables. Cada uno de estos parámetros se modificará ligeramente durante el entrenamiento, y entre todos ellos capturará los patrones necesarios para clasificar flores correctamente.

Esta cantidad de parámetros es modesta comparada con las redes modernas que tienen millones o miles de millones de parámetros, pero ilustra el principio fundamental: cada conexión entre neuronas es un parámetro que la red puede ajustar para aprender. Cuantas más neuronas y conexiones, más capacidad tiene la red para capturar patrones complejos, pero también más datos necesita para evitar sobreajustar.

### Visualizando la arquitectura

Podemos pensar en la red como una tubería de información que se transforma en cada etapa. La entrada es un vector de 4 números que representa una flor. Después de la primera capa oculta, se convierte en un vector de 16 números que representa características de bajo nivel. Después de la segunda capa oculta, se convierte en un vector de 8 números que representa características de nivel medio. Finalmente, la capa de salida lo transforma en un vector de 3 probabilidades.

In [ ]:
# Dimensiones de cada capa
input_size = 4      # características: sépalo largo/ancho, pétalo largo/ancho
hidden1_size = 16   # primera capa oculta
hidden2_size = 8    # segunda capa oculta
output_size = 3     # clases: setosa, versicolor, virginica

# shapes de los pesos
W1 = np.random.randn(input_size, hidden1_size)    # (4, 16)
b1 = np.random.randn(1, hidden1_size)              # (1, 16)
W2 = np.random.randn(hidden1_size, hidden2_size)  # (16, 8)
b2 = np.random.randn(1, hidden2_size)              # (1, 8)
W3 = np.random.randn(hidden2_size, output_size)   # (8, 3)
b3 = np.random.randn(1, output_size)               # (1, 3)

print(f"Pesos W1: {W1.shape} → {input_size*hidden1_size} parámetros")
print(f"Pesos W2: {W2.shape} → {hidden1_size*hidden2_size} parámetros")
print(f"Pesos W3: {W3.shape} → {hidden2_size*output_size} parámetros")
print(f"Total parámetros: {input_size*hidden1_size + hidden1_size*hidden2_size + hidden2_size*output_size + hidden1_size + hidden2_size + output_size}")

Con la arquitectura definida, estamos listos para ver cómo los datos fluyen a través de estas capas. En la siguiente sección implementaremos la propagación hacia adelante, calculando paso a paso cómo una flor de entrada se transforma en una predicción de salida.

# 5. Forward propagation: el flujo hacia adelante

La propagación hacia adelante es el proceso mediante el cual los datos de entrada atraviesan la red neuronal para producir una predicción. Es como verter harina en una máquina de pan: entra por un lado, pasa por varios procesos de transformación, y sale como pan horneado. En cada etapa, la información se transforma, se combina y se refina hasta que obtenemos el resultado final. No hay magia aquí, solo multiplicaciones matriciales y funciones de activación aplicadas sistemáticamente.

### El proceso paso a paso

Imaginemos que tenemos una flor del dataset Iris, con sus cuatro características ya normalizadas. Esta flor es nuestra entrada, un vector de cuatro números. El proceso de forward propagation la tomará y la transformará a través de tres etapas de cálculo hasta obtener tres probabilidades que suman a uno.

La primera etapa ocurre en la primera capa oculta. Cada una de las 16 neuronas de esta capa recibe las cuatro características de la flor, las multiplica por sus pesos correspondientes, suma todo junto con su sesgo, y aplica la función ReLU. Pero en lugar de calcular esto neurona por neurona en un bucle lento, usamos multiplicación de matrices que realiza todos estos cálculos simultáneamente. Si la entrada es un vector fila de shape (1, 4) y la matriz de pesos W1 tiene shape (4, 16), entonces la operación entrada @ W1 produce directamente un vector de shape (1, 16) con las sumas ponderadas de las 16 neuronas. Luego sumamos el vector de sesgos b1 y aplicamos ReLU elemento por elemento.

La segunda etapa ocurre en la segunda capa oculta. El vector de 16 activaciones que salió de la primera capa ahora se convierte en la entrada de la segunda. Multiplicamos por la matriz W2 de shape (16, 8), sumamos los sesgos b2, y aplicamos ReLU nuevamente. Obtenemos un vector de 8 activaciones que representa características de nivel superior sobre la flor.

La tercera etapa es la capa de salida. Las 8 activaciones de la segunda capa se multiplican por la matriz W3 de shape (8, 3), se suman los sesgos b3, y obtenemos tres valores que todavía no son probabilidades. Estos tres valores se llaman logits, y para convertirlos en probabilidades que sumen a uno aplicamos la función Softmax.

### La matemática en detalle

Para un solo ejemplo de entrada x con shape (1, 4), las operaciones son las siguientes:

**Primera capa oculta:**
- z1 = x @ W1 + b1 → shape (1, 16)
- a1 = ReLU(z1) → shape (1, 16)

**Segunda capa oculta:**
- z2 = a1 @ W2 + b2 → shape (1, 8)
- a2 = ReLU(z2) → shape (1, 8)

**Capa de salida:**
- z3 = a2 @ W3 + b3 → shape (1, 3)
- a3 = Softmax(z3) → shape (1, 3)

El símbolo @ denota multiplicación matricial. Cuando multiplicamos un vector fila por una matriz, el resultado es otro vector fila donde cada elemento es el producto punto del vector original con una columna de la matriz. Esto es exactamente lo que queremos: cada neurona de la siguiente capa combina todas las salidas de la capa anterior con sus propios pesos.

La función Softmax merece atención especial porque es la que convierte los logits en probabilidades interpretables. Para un vector z de tres elementos, Softmax define la probabilidad de la clase i como eᶻᶦ dividido entre la suma de todas las eᶻʲ. Esta exponenciación tiene un efecto importante: amplifica las diferencias. Si un logit es ligeramente mayor que los otros, después de Softmax su probabilidad será sustancialmente mayor. Esto significa que la predicción final no solo indica qué clase es más probable, sino también cuán confiada está la red en esa elección.

### Implementación completa en NumPy

Veamos cómo implementar forward propagation en código. Mostraremos cada paso explícitamente para que puedas ver exactamente qué está sucediendo.

In [ ]:
import numpy as np

def relu(z):
    """Función de activación ReLU"""
    return np.maximum(0, z)

def softmax(z):
    """Softmax para clasificación multiclase"""
    # Restamos el máximo para estabilidad numérica
    exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)

def forward_propagation(X, W1, b1, W2, b2, W3, b3):
    """
    Forward propagation completa para 4→16→8→3

    Parámetros:
    -----------
    X : array de shape (n_samples, 4) - características de entrada
    W1, b1 : pesos y sesgos de primera capa (4, 16) y (1, 16)
    W2, b2 : pesos y sesgos de segunda capa (16, 8) y (1, 8)
    W3, b3 : pesos y sesgos de capa de salida (8, 3) y (1, 3)

    Retorna:
    --------
    a3 : probabilidades de salida, shape (n_samples, 3)
    cache : diccionario con todos los valores intermedios
    """
    # Primera capa oculta
    z1 = X @ W1 + b1
    a1 = relu(z1)

    # Segunda capa oculta
    z2 = a1 @ W2 + b2
    a2 = relu(z2)

    # Capa de salida
    z3 = a2 @ W3 + b3
    a3 = softmax(z3)

    # Guardamos todo para backpropagation
    cache = {'X': X, 'z1': z1, 'a1': a1, 'z2': z2, 'a2': a2, 'z3': z3, 'a3': a3}

    return a3, cache

# Inicialización de pesos con Xavier para mejor convergencia
np.random.seed(42)
W1 = np.random.randn(4, 16) * np.sqrt(2.0 / 4)
b1 = np.zeros((1, 16))
W2 = np.random.randn(16, 8) * np.sqrt(2.0 / 16)
b2 = np.zeros((1, 8))
W3 = np.random.randn(8, 3) * np.sqrt(2.0 / 8)
b3 = np.zeros((1, 3))

# Probemos con una flor de ejemplo
X_sample = X_test[0:1]  # Una flor del conjunto de prueba
print(f"Entrada (4 características normalizadas): {X_sample[0]}")
print(f"Shape de entrada: {X_sample.shape}")

# Forward propagation
probs, cache = forward_propagation(X_sample, W1, b1, W2, b2, W3, b3)

print(f"\nDespués de capa 1 (16 neuronas):")
print(f"  Activaciones min: {cache['a1'].min():.4f}, max: {cache['a1'].max():.4f}")

print(f"\nDespués de capa 2 (8 neuronas):")
print(f"  Activaciones min: {cache['a2'].min():.4f}, max: {cache['a2'].max():.4f}")

print(f"\nProbabilidades de salida:")
print(f"  Setosa: {probs[0, 0]:.4f}")
print(f"  Versicolor: {probs[0, 1]:.4f}")
print(f"  Virginica: {probs[0, 2]:.4f}")
print(f"  Suma: {probs.sum():.4f} ✓")

La salida de este código muestra el flujo de información a través de las capas. La entrada normalizada se transforma en activaciones de la primera capa, luego en activaciones de la segunda capa, y finalmente en tres probabilidades que suman exactamente a uno. El predicted_class sería argmax(probs), es decir, el índice de la probabilidad más alta.

### ¿Qué nos dice la salida?

Las tres probabilidades outputeadas por la red son nuestra predicción. Si la red está bien entrenada, la probabilidad más alta corresponderá a la clase correcta de la flor. Pero estas probabilidades también indican la confianza de la red: si una flor es claramente setosa, la red outputeará algo como [0.95, 0.04, 0.01], mostrando alta confianza. Si la flor está en la frontera entre versicolor y virginica, podríamos ver algo como [0.02, 0.55, 0.43], indicando que la red está más incertaine.

El forward propagation es solo la primera mitad del proceso de aprendizaje. La red ha hecho una predicción, pero no sabe si es correcta o no. Para eso necesitamos medir el error, y para eso necesitamos definir una función de pérdida. En la siguiente sección veremos cómo cuantificar «qué tan equivocada» está la predicción, y luego en la sección de backpropagation aprenderemos cómo usar esa información para mejorar los pesos.

# 7. Backward propagation: el alma del aprendizaje

La propagación hacia atrás es el algoritmo que hace posible el aprendizaje en redes neuronales. Es la respuesta a la pregunta: «dado que la red cometió un error, ¿cómo ajustamos los pesos para no cometerlo de nuevo?» Sin backpropagation, tendríamos pesos que se actualizan aleatoriamente, hoping por coincidencia encontrar los valores correctos. Con backpropagation, cada peso sabe exactamente cuánto contributed al error total y en qué dirección debe ajustarse. Este algoritmo, combinado con Gradient Descent, es lo que permite a las redes neuronales aprender de manera eficiente de miles o millones de ejemplos.

### El problema que resuelve backpropagation

Imaginemos que nuestra red ha hecho una predicción y la pérdida es alta. Sabemos que el error viene de los pesos: si los pesos fueran diferentes, la predicción habría sido diferente. Pero hay 243 parámetros en nuestra red, cada uno contribuyendo de manera compleja a la predicción final. No podemos simplemente «adivinar» qué peso cambiar y en qué dirección. Necesitamos saber, para cada peso individual, dos cosas: si aumentar o disminuir ese peso reduce el error, y cuánto debe ser el ajuste.

La respuesta a estas preguntas está en las derivadas. La derivada parcial de la pérdida respecto a un peso nos dice exactamente cuánto cambia la pérdida cuando modificamos ese peso. Si la derivada es positiva, aumentar el peso aumenta la pérdida, así que debemos disminuirlo. Si la derivada es negativa, aumentar el peso disminuye la pérdida, así que debemos aumentarlo. Y el tamaño de la derivada nos indica la magnitud del ajuste necesario.

El desafío es calcular estas derivadas. La pérdida depende de la predicción, la predicción depende de la capa de salida, la capa de salida depende de la segunda capa oculta, y así sucesivamente hasta llegar a los pesos de la primera capa. Tenemos una cadena de dependencias: Loss ← predicción ← z₃ ← a₂ ← z₂ ← a₁ ← pesos. Para encontrar cómo la pérdida depende de los pesos del inicio de la cadena, necesitamos traversing toda la cadena en dirección inversa, multiplicando derivadas en cada paso. Esto es exactamente lo que hace la regla de la cadena del cálculo.

### La regla de la cadena, explicada suavemente

La regla de la cadena es un teorema del cálculo que nos dice cómo derivar funciones compuestas. Si tenemos una función f que depende de g, y g depende de x, entonces la derivada de f respecto a x es la derivada de f respecto a g multiplicada por la derivada de g respecto a x. En símbolos: df/dx = df/dg · dg/dx.

En redes neuronales, tenemos cadenas largas de funciones compuestas. La pérdida depende de las probabilidades, las probabilidades dependen de z₃, z₃ depende de a₂, a₂ depende de z₂, z₂ depende de a₁, a₁ depende de z₁, y z₁ depende de los pesos. La derivada de la pérdida respecto a un peso es el producto de todas las derivadas intermedias a lo largo de esta cadena.

La intuición clave es que cada factor en este producto representa cuánto «influye» una variable en la siguiente. Si cambiar z₂ apenas afecta la pérdida (derivada pequeña), entonces todo lo que depende de z₂, incluyendo los pesos que producen z₂, apenas affectará la pérdida. Por otro lado, si cambiar a₂ afecta significativamente la pérdida, entonces los pesos que producen a₂ deben ajustarse más, proporcionalmente a esa influencia.

### Un ejemplo numérico completo

La mejor manera de entender backpropagation es ver un ejemplo completo paso a paso. Usaremos una sola flor del dataset, con la etiqueta correcta siendo versicolor (clase 1). Para simplificar, mostraremos el cálculo para un ejemplo individual, aunque en la práctica promediamos sobre batches.

**Datos de entrada:**
- Características normalizadas: x = [0.5, -0.2, 1.1, -0.3]
- Etiqueta one-hot: y = [0, 1, 0] (versicolor)
- Arquitectura: 4 → 16 → 8 → 3

**Forward propagation (resultados guardados en cache):**
```
z1 = x @ W1 + b1      # shape (16,)
a1 = ReLU(z1)         # shape (16,)

z2 = a1 @ W2 + b2     # shape (8,)
a2 = ReLU(z2)         # shape (8,)

z3 = a2 @ W3 + b3     # shape (3,)
probs = softmax(z3)   # shape (3,) → [0.05, 0.90, 0.05]
```

Supongamos que la predicción fue [0.05, 0.90, 0.05], lo cual es correcto pero no perfectamente confiado. La pérdida será -log(0.90) ≈ 0.105.

**Paso 1: Gradiente en la capa de salida**

Comenzamos desde el final. La derivada de la entropía cruzada combinada con Softmax tiene una forma simplicada: para la capa de salida, el gradiente es simplemente probs - y. Esto es uno de los resultados más útiles del deep learning: no necesitamos calcular derivadas complicadas de Softmax por separado.

```python
# Gradiente de la pérdida respecto a z3
d_z3 = probs - y  # shape (3,)
# d_z3 = [0.05, 0.90, 0.05] - [0, 1, 0] = [0.05, -0.10, 0.05]
```

Este vector d_z3 nos dice cómo ajustar los sesgos y pesos de la capa de salida. Los valores positivos indican que z3 debería aumentar (para aumentar la probabilidad de esa clase), y los valores negativos indican que debería disminuir.

**Paso 2: Gradientes de W3 y b3**

Para calcular el gradiente respecto a los pesos W3, recordamos que z3 = a2 @ W3 + b3. La derivada de z3 respecto a W3 involucra la transpuesta de a2. Intuitivamente, cada peso W3[i,j] conecta la neurona j de la capa oculta con la neurona i de la salida, y su gradiente es el producto del error en la salida (d_z3[i]) por la activación de entrada (a2[j]).

```python
# Gradiente de W3: cada peso se actualiza por error × activación de entrada
d_W3 = a2.T @ d_z3  # shape (8, 3)

# Gradiente de b3: suma del error (promediado sobre el batch)
d_b3 = np.sum(d_z3, axis=0, keepdims=True)  # shape (1, 3)
```

**Paso 3: Gradiente que fluye hacia la segunda capa oculta**

Ahora necesitamos propagar el error hacia atrás a través de la segunda capa oculta. El error que llega a a2 es d_z3 @ W3.T. Pero este error todavía está en el espacio de activaciones, y necesitamos convertirlo al espacio de z2 para actualizar los pesos de la segunda capa. Para eso multiplicamos por la derivada de ReLU.

La derivada de ReLU es simple: es 1 donde z2 > 0 y 0 donde z2 ≤ 0. Esto significa que las neuronas que están «apagadas» (z2 negativo, activación cero) no reciben ningún gradiente: su salida no contribuyó a la predicción, así que ajustar sus pesos no cambiaría la pérdida.

```python
# Error que llega a a2 desde la capa de salida
d_a2 = d_z3 @ W3.T  # shape (8,)

# Derivada de ReLU: 1 donde z2 > 0, 0 donde z2 ≤ 0
d_relu = (z2 > 0).astype(float)  # shape (8,)

# Error en z2
d_z2 = d_a2 * d_relu  # Solo las neuronas activas reciben gradiente
```

**Paso 4: Gradientes de W2 y b2**

Con el gradiente en z2, podemos calcular los gradientes de los pesos de la segunda capa exactamente como hicimos con W3.

```python
# Gradiente de W2
d_W2 = a1.T @ d_z2  # shape (16, 8)

# Gradiente de b2
d_b2 = np.sum(d_z2, axis=0, keepdims=True)  # shape (1, 8)
```

**Paso 5: Gradiente que fluye hacia la primera capa oculta**

Repetimos el proceso para la primera capa oculta. El error fluye desde z2 hacia a1, y desde a1 hacia z1.

```python
# Error que llega a a1 desde la segunda capa
d_a1 = d_z2 @ W2.T  # shape (16,)

# Error en z1
d_z1 = d_a1 * d_relu_1  # donde d_relu_1 = (z1 > 0).astype(float)
```

**Paso 6: Gradientes de W1 y b1**

Finalmente, llegamos a los pesos de la primera capa.

```python
# Gradiente de W1
d_W1 = X.T @ d_z1  # shape (4, 16)

# Gradiente de b1
d_b1 = np.sum(d_z1, axis=0, keepdims=True)  # shape (1, 16)
```

### Actualización de pesos con Gradient Descent

Una vez que tenemos todos los gradientes, la actualización es simple. Restamos el gradiente (multiplicado por el learning rate) de los pesos actuales. El learning rate es un hiperparámetro que controla qué tan grandes son los pasos que damos en la dirección del gradiente.

```python
learning_rate = 0.01

# Actualizar pesos
W3 = W3 - learning_rate * d_W3
b3 = b3 - learning_rate * d_b3
W2 = W2 - learning_rate * d_W2
b2 = b2 - learning_rate * d_b2
W1 = W1 - learning_rate * d_W1
b1 = b1 - learning_rate * d_b1
```

El signo negativo es crucial: si el gradiente es positivo, el peso disminuye; si es negativo, el peso aumenta. En ambos casos, el cambio es en la dirección que reduce la pérdida.

### Implementación completa de backpropagation

In [ ]:
def backward_propagation(X, y, cache, W1, W2, W3):
    """
    Backpropagation completa para 4→16→8→3
    """
    m = X.shape[0]  # número de ejemplos

    a1, a2, a3 = cache['a1'], cache['a2'], cache['a3']
    z1, z2, z3 = cache['z1'], cache['z2'], cache['z3']

    # === Capa de salida ===
    # d_z3 = probs - y (derivada combinada Softmax + CrossEntropy)
    d_z3 = a3 - y

    # Gradientes de W3 y b3
    d_W3 = a2.T @ d_z3 / m
    d_b3 = np.sum(d_z3, axis=0, keepdims=True) / m

    # === Segunda capa oculta ===
    # Error fluyendo hacia atrás
    d_a2 = d_z3 @ W3.T
    d_z2 = d_a2 * (z2 > 0).astype(float)  # Derivada de ReLU

    # Gradientes de W2 y b2
    d_W2 = a1.T @ d_z2 / m
    d_b2 = np.sum(d_z2, axis=0, keepdims=True) / m

    # === Primera capa oculta ===
    d_a1 = d_z2 @ W2.T
    d_z1 = d_a1 * (z1 > 0).astype(float)  # Derivada de ReLU

    # Gradientes de W1 y b1
    d_W1 = X.T @ d_z1 / m
    d_b1 = np.sum(d_z1, axis=0, keepdims=True) / m

    gradients = {
        'd_W1': d_W1, 'd_b1': d_b1,
        'd_W2': d_W2, 'd_b2': d_b2,
        'd_W3': d_W3, 'd_b3': d_b3
    }

    return gradients

### La intuición final

Backpropagation funciona porque la regla de la cadena permite descomponer el efecto de cada peso en el error final. El gradiente que calculamos para W1 le dice a ese peso específico: «tu contribución al error fue de X cantidad, así que ajústate en dirección negativa para reducir la pérdida». Cuando hacemos esto para todos los pesos simultáneamente, la red mejora incrementalmente en cada ejemplo de entrenamiento.

El proceso completo —forward propagation para obtener una predicción, cálculo de la pérdida, backpropagation para obtener los gradientes, y actualización de pesos— constituye una sola iteración de entrenamiento. Repitiendo este proceso miles de veces sobre el dataset, los pesos convergen hacia valores que hacen predicciones precisas. En la siguiente sección, pondremos todo junto en la implementación completa y veremos cómo la red aprende en la práctica.

# 6. Midiendo el error: la función de pérdida

La red neuronal ha hecho su trabajo: ha recibido las características de una flor y ha outputeado tres probabilidades. Pero hasta ahora no tenemos forma de saber si la predicción es buena o mala. Necesitamos una métrica que cuantifique el error, algo que la red pueda minimizar durante el entrenamiento. Esta métrica es la función de pérdida, y su correcta elección es fundamental para que el aprendizaje funcione.

### ¿Qué es la función de pérdida?

La función de pérdida es una fórmula que toma dos cosas: la predicción de la red y la respuesta correcta, y devuelve un número único que representa «qué tan equivocada» está la predicción. Si la predicción es perfecta, la pérdida es cero. Si la predicción es completamente errónea, la pérdida es alta. Durante el entrenamiento, queremos que esta pérdida disminuya, lo cual significa que la red está aprendiendo a hacer mejores predicciones.

Hay muchas funciones de pérdida posibles, cada una apropiada para diferentes tipos de problemas. Para regresión, donde la salida es un número continuo, se usa típicamente el error cuadrático medio. Para clasificación binaria, donde solo hay dos clases, se usa la entropía cruzada binaria. Para clasificación multiclase, que es nuestro caso con las tres especies de Iris, usamos la entropía cruzada categórica. El nombre «entropía cruzada» viene de la teoría de información, pero la intuición es más simple de lo que parece.

### La fórmula y su intuición

La entropía cruzada categórica para un solo ejemplo se define como:

**L = -Σ yᵢ · log(ŷᵢ)**

Donde yᵢ es la probabilidad real (1 para la clase correcta, 0 para las demás) y ŷᵢ es la probabilidad predicha por la red. La suma se toma sobre las tres clases.

Para entender por qué esta fórmula tiene sentido, consideremos un ejemplo. Supongamos que la flor de entrada es una versicolor (clase 1), y la red ha predicho ŷ = [0.05, 0.90, 0.05]. En codificación one-hot, y = [0, 1, 0]. La pérdida es:

L = -(0 · log(0.05) + 1 · log(0.90) + 0 · log(0.05)) = -log(0.90) ≈ 0.105

La pérdida es baja, como esperado, porque la red predijo correctamente con alta confianza.

Ahora consideremos el caso donde la red se equivocó completamente: predijo ŷ = [0.90, 0.05, 0.05] cuando la respuesta correcta era versicolor. La pérdida sería:

L = -(0 · log(0.90) + 1 · log(0.05) + 0 · log(0.05)) = -log(0.05) ≈ 3.00

La pérdida es mucho mayor. Notemos que solo el término correspondiente a la clase correcta contribuye a la pérdida: los ceros en y hacen que los otros términos se anulen. Esto tiene sentido: solo nos importa qué tan confiada está la red en la respuesta correcta, no en las respuestas incorrectas.

### La magia de log

La función log hace algo interesante en esta fórmula. Cuando la red predice con alta confianza la clase correcta (ŷ ≈ 1), log(1) = 0 y la pérdida es cercana a cero. Cuando la red predice con baja confianza la clase correcta (ŷ ≈ 0), log(0) va hacia menos infinito y la pérdida se dispara. Esto penaliza severamente las predicciones confiadas pero incorrectas, exactamente lo que queremos.

La entropía cruzada también tiene una propiedad matemática conveniente cuando se combina con Softmax en la capa de salida: la derivada se simplifica dramáticamente. En lugar de calcular la derivada complicada de Softmax multiplicada por la derivada de la entropía cruzada, la derivada combinada es simplemente ŷ - y. Esta simplificación es una de las razones por las que Softmax + entropía cruzada es la combinación estándar para clasificación.

### Pérdida para todo el dataset

Hasta ahora hemos calculado la pérdida para una sola flor. Para el dataset completo, promediamos la pérdida de todos los ejemplos. Si tenemos N flores, la pérdida total es:

**L_total = (1/N) · Σ Lᵢ**

Esta pérdida promedio es lo que la red intentará minimizar durante el entrenamiento. Un valor bajo de pérdida total significa que, en promedio, la red está haciendo buenas predicciones con alta confianza en las respuestas correctas.

In [ ]:
def categorical_cross_entropy(y_true, y_pred):
    """
    Calcula la entropía cruzada categórica.

    Parámetros:
    -----------
    y_true : array de shape (n_samples, 3) - one-hot encoded
    y_pred : array de shape (n_samples, 3) - probabilidades

    Retorna:
    --------
    loss : float - pérdida promedio
    """
    # Evitar log(0) agregando un epsilon pequeño
    epsilon = 1e-15
    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)

    # Calcular entropía cruzada
    loss = -np.sum(y_true * np.log(y_pred), axis=1)

    return np.mean(loss)

# Convertir etiquetas a one-hot
def one_hot(y, num_classes):
    n = len(y)
    one_hot = np.zeros((n, num_classes))
    one_hot[np.arange(n), y] = 1
    return one_hot

y_test_onehot = one_hot(y_test, 3)

# Calcular pérdida para un batch
sample_probs = np.array([[0.05, 0.90, 0.05],   # Correcto
                         [0.90, 0.05, 0.05],   # Incorrecto
                         [0.02, 0.48, 0.50]])  # Incierto

sample_labels = one_hot(np.array([1, 0, 2]), 3)

loss = categorical_cross_entropy(sample_labels, sample_probs)
print(f"Pérdida del batch: {loss:.4f}")
print("  Ejemplo 1 (correcto, confiado): pérdida baja")
print("  Ejemplo 2 (incorrecto, confiado): pérdida alta")
print("  Ejemplo 3 (incierto): pérdida moderada")

Con la función de pérdida definida, tenemos todo lo necesario para medir el error de la red. Pero saber que la red se equivoca no nos dice cómo corregirla. Para eso necesitamos entender cómo cada peso individual contribuye al error total, y eso es exactamente lo que hace la propagación hacia atrás. En la siguiente sección desempacaremos este algoritmo y verá por qué la regla de la cadena del cálculo es la pieza central del aprendizaje en redes neuronales.

# 8. Implementación completa en NumPy

Ha llegado el momento de unir todas las piezas que hemos discutido en código funcional. En esta sección presentaremos la implementación completa de nuestra red neuronal 4→16→8→3, desde la inicialización de pesos hasta el bucle de entrenamiento. El código que sigue es todo lo que necesitas para entrenar una red neuronal desde cero, sin dependencias externas más allá de NumPy.

La implementación está organizada en funciones claramente separadas para cada operación: inicialización, forward propagation, backward propagation, cálculo de pérdida, predicción y entrenamiento. Esta modularidad no es solo buena práctica de programación, sino que refleja la estructura conceptual de una red neuronal donde cada componente tiene una responsabilidad específica.

### Inicialización de pesos

La inicialización de los pesos es más sutil de lo que podría parecer. Si inicializamos todos los pesos a cero, todas las neuronas de una capa harán exactamente lo mismo y la red no podrá aprender patrones diversos. Si inicializamos con valores demasiado grandes, las activaciones saturan y los gradientes explotan. Si inicializamos con valores demasiado pequeños, las activaciones se atenúan y los gradientes vanish.

La inicialización de Xavier (también llamada Glorot) resuelve este problema escalando los pesos según el tamaño de las capas que conectan. Para una capa con n entradas, los pesos se inicializan con una distribución normal de media cero y varianza 2/n. Esto mantiene la varianza de las activaciones estable a través de las capas, permitiendo que el gradiente fluya sin atenuarse ni explosionar excesivamente.

In [ ]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

def initialize_parameters(input_size, hidden1_size, hidden2_size, output_size):
    """
    Inicializa pesos con la técnica de Xavier para mejor convergencia.
    """
    np.random.seed(42)  # Reproducibilidad

    # Primera capa: 4 -> 16
    W1 = np.random.randn(input_size, hidden1_size) * np.sqrt(2.0 / input_size)
    b1 = np.zeros((1, hidden1_size))

    # Segunda capa: 16 -> 8
    W2 = np.random.randn(hidden1_size, hidden2_size) * np.sqrt(2.0 / hidden1_size)
    b2 = np.zeros((1, hidden2_size))

    # Capa de salida: 8 -> 3
    W3 = np.random.randn(hidden2_size, output_size) * np.sqrt(2.0 / hidden2_size)
    b3 = np.zeros((1, output_size))

    parameters = {
        'W1': W1, 'b1': b1,
        'W2': W2, 'b2': b2,
        'W3': W3, 'b3': b3
    }

    return parameters

# Definir dimensiones de la arquitectura
input_size = 4      # características del Iris
hidden1_size = 16   # primera capa oculta
hidden2_size = 8    # segunda capa oculta
output_size = 3     # clases: setosa, versicolor, virginica

# Inicializar parámetros
params = initialize_parameters(input_size, hidden1_size, hidden2_size, output_size)

print("Pesos inicializados:")
print(f"  W1: {params['W1'].shape} - conecta entrada con primera capa oculta")
print(f"  W2: {params['W2'].shape} - conecta primera con segunda capa oculta")
print(f"  W3: {params['W3'].shape} - conecta segunda capa con salida")
print(f"\nTotal parámetros: {sum(p.size for p in params.values())}")

### Funciones de activación y sus derivadas

Implementamos las funciones de activación y sus derivadas de manera eficiente, aprovechando las operaciones vectorizadas de NumPy.

In [ ]:
def relu(z):
    """Función de activación ReLU"""
    return np.maximum(0, z)

def relu_derivative(z):
    """Derivada de ReLU: 1 donde z > 0, 0 donde z <= 0"""
    return (z > 0).astype(float)

def softmax(z):
    """
    Softmax con estabilidad numérica.
    Resta el máximo para evitar overflow en la exponenciación.
    """
    exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)

def categorical_cross_entropy(y_true, y_pred):
    """
    Calcula la pérdida de entropía cruzada categórica.
    """
    epsilon = 1e-15
    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
    loss = -np.sum(y_true * np.log(y_pred), axis=1)
    return np.mean(loss)

### Forward propagation completo

Esta función encapsula todo el flujo de datos desde la entrada hasta la predicción, guardando los valores intermedios necesarios para backpropagation.

In [ ]:
def forward_propagation(X, params):
    """
    Forward propagation completa.

    Retorna:
    --------
    A3 : probabilidades de salida
    cache : diccionario con todos los valores intermedios
    """
    W1, b1 = params['W1'], params['b1']
    W2, b2 = params['W2'], params['b2']
    W3, b3 = params['W3'], params['b3']

    # Primera capa oculta
    Z1 = X @ W1 + b1
    A1 = relu(Z1)

    # Segunda capa oculta
    Z2 = A1 @ W2 + b2
    A2 = relu(Z2)

    # Capa de salida
    Z3 = A2 @ W3 + b3
    A3 = softmax(Z3)

    cache = {
        'X': X, 'Z1': Z1, 'A1': A1,
        'Z2': Z2, 'A2': A2, 'Z3': Z3, 'A3': A3
    }

    return A3, cache

### Backward propagation completo

Esta es la función que implementa todo el algoritmo de propagación hacia atrás que discutimos en la sección anterior.

In [ ]:
def backward_propagation(X, y, params, cache):
    """
    Backward propagation completa.

    Retorna:
    --------
    gradients : diccionario con los gradientes de todos los parámetros
    """
    m = X.shape[0]  # número de ejemplos en el batch

    A1, A2, A3 = cache['A1'], cache['A2'], cache['A3']
    Z1, Z2 = cache['Z1'], cache['Z2']

    W1, W2, W3 = params['W1'], params['W2'], params['W3']

    # === Capa de salida ===
    # Derivada combinada de Softmax + CrossEntropy: A3 - y
    dZ3 = A3 - y

    dW3 = A2.T @ dZ3 / m
    db3 = np.sum(dZ3, axis=0, keepdims=True) / m

    # === Segunda capa oculta ===
    dA2 = dZ3 @ W3.T
    dZ2 = dA2 * relu_derivative(Z2)

    dW2 = A1.T @ dZ2 / m
    db2 = np.sum(dZ2, axis=0, keepdims=True) / m

    # === Primera capa oculta ===
    dA1 = dZ2 @ W2.T
    dZ1 = dA1 * relu_derivative(Z1)

    dW1 = X.T @ dZ1 / m
    db1 = np.sum(dZ1, axis=0, keepdims=True) / m

    gradients = {
        'dW1': dW1, 'db1': db1,
        'dW2': dW2, 'db2': db2,
        'dW3': dW3, 'db3': db3
    }

    return gradients

### Función de predicción

In [ ]:
def predict(X, params):
    """
    Retorna predicciones de clase (no probabilidades).
    """
    probabilities, _ = forward_propagation(X, params)
    return np.argmax(probabilities, axis=1)

### El bucle de entrenamiento

El bucle de entrenamiento une todas las piezas en el ciclo completo de aprendizaje: forward propagation para obtener la predicción, cálculo de la pérdida, backward propagation para obtener los gradientes, y actualización de pesos para mejorar.

In [ ]:
def train_network(X_train, y_train, X_test, y_test, params,
                  learning_rate=0.1, epochs=1000, print_every=100):
    """
    Entrena la red neuronal y registra el progreso.
    """
    history = {
        'train_loss': [], 'test_loss': [],
        'train_acc': [], 'test_acc': []
    }

    for epoch in range(epochs):
        # Forward propagation
        A3, cache = forward_propagation(X_train, params)

        # Calcular pérdida
        loss = categorical_cross_entropy(y_train, A3)

        # Backward propagation
        gradients = backward_propagation(X_train, y_train, params, cache)

        # Actualizar parámetros (Gradient Descent)
        for key in params:
            # Ensure key for bias also exists in gradients (e.g., 'b1' vs 'db1')
            if 'd' + key in gradients:
                params[key] -= learning_rate * gradients['d' + key]

        # Evaluar cada certain número de épocas
        if (epoch + 1) % print_every == 0:
            # Pérdida en train
            train_probs, _ = forward_propagation(X_train, params)
            train_loss = categorical_cross_entropy(y_train, train_probs)
            train_acc = np.mean(np.argmax(train_probs, axis=1) == np.argmax(y_train, axis=1))

            # Pérdida en test
            test_probs, _ = forward_propagation(X_test, params)
            test_loss = categorical_cross_entropy(y_test, test_probs)
            test_acc = np.mean(np.argmax(test_probs, axis=1) == np.argmax(y_test, axis=1))

            history['train_loss'].append(train_loss)
            history['test_loss'].append(test_loss)
            history['train_acc'].append(train_acc)
            history['test_acc'].append(test_acc)

            print(f"Época {epoch+1:4d}/{epochs} | "
                  f"Pérdida: {train_loss:.4f} | "
                  f"Accuracy: {train_acc*100:.1f}% | "
                  f"Test Acc: {test_acc*100:.1f}%")

    return params, history

### Entrenando nuestra red

Con todas las funciones definidas, podemos entrenar la red con solo unas pocas líneas de código. Cargamos los datos, los preparamos, inicializamos la red y ejecutamos el entrenamiento.

In [ ]:
# Cargar y preparar datos
iris = load_iris()
X = iris.data
y = iris.target

# One-hot encoding
def one_hot(y, num_classes):
    n = len(y)
    encoded = np.zeros((n, num_classes))
    encoded[np.arange(n), y] = 1
    return encoded

y_onehot = one_hot(y, 3)

# División train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y_onehot, test_size=0.2, random_state=42, stratify=y
)

# Normalización
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"Conjunto de entrenamiento: {X_train.shape[0]} flores")
print(f"Conjunto de prueba: {X_test.shape[0]} flores")
print("\nEntrenando red neuronal 4→16→8→3...\n")

# Entrenar
trained_params, history = train_network(
    X_train, y_train, X_test, y_test,
    params, learning_rate=0.5, epochs=2000, print_every=200
)

# Resultados finales
final_train_acc = history['train_acc'][-1]
final_test_acc = history['test_acc'][-1]
print(f"\n{'='*50}")
print(f"RESULTADOS FINALES")
print(f"{'='*50}")
print(f"Accuracy en entrenamiento: {final_train_acc*100:.2f}%")
print(f"Accuracy en prueba: {final_test_acc*100:.2f}%")

La salida de este entrenamiento mostrará cómo la pérdida disminuye y la precisión aumenta a lo largo de las épocas. Al final, deberíamos ver una precisión superior al 90% en el conjunto de prueba, demostrando que nuestra implementación desde cero funciona correctamente. En la siguiente sección visualizaremos el proceso de aprendizaje y analizaremos los resultados más detalladamente.

# 9. Entrenamiento y resultados

Ha llegado el momento de ver nuestra red neuronal en acción. En la sección anterior implementamos todo el código necesario para entrenar la red, pero no vimos qué sucede durante el proceso de aprendizaje. En esta sección analizaremos las curvas de entrenamiento, observaremos cómo la pérdida disminuye y la precisión aumenta, y examinaremos predicciones específicas para verificar que la red está aprendiendo los patrones correctos del dataset Iris.

El entrenamiento de una red neuronal no es lineal: hay momentos de progreso rápido, mesetas donde parece que nada mejora, y ocasionalmente retrocesos. Visualizar estas métricas nos permite diagnosticar el estado del entrenamiento y asegurarnos de que todo va según lo esperado. Si la pérdida no disminuye, algo está mal en la implementación o en los hiperparámetros. Si la pérdida de entrenamiento disminuye pero la de prueba aumenta, estamos sobreajustando el modelo a los datos de entrenamiento.

### Visualizando las curvas de aprendizaje

Usaremos matplotlib para graficar la evolución de la pérdida y la precisión a lo largo de las épocas. Estas visualizaciones son herramientas diagnósticas esenciales que todo practicante de machine learning debe saber interpretar.

In [ ]:
import matplotlib.pyplot as plt

def plot_training_history(history):
    """
    Visualiza las curvas de entrenamiento.
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Curva de pérdida
    axes[0].plot(history['train_loss'], label='Entrenamiento', linewidth=2, color='#2196F3')
    axes[0].plot(history['test_loss'], label='Prueba', linewidth=2, color='#FF5722')
    axes[0].set_xlabel('Épocas', fontsize=12)
    axes[0].set_ylabel('Pérdida (Cross-Entropy)', fontsize=12)
    axes[0].set_title('Evolución de la Pérdida', fontsize=14)
    axes[0].legend(fontsize=10)
    axes[0].grid(True, alpha=0.3)

    # Curva de precisión
    axes[1].plot(history['train_acc'], label='Entrenamiento', linewidth=2, color='#2196F3')
    axes[1].plot(history['test_acc'], label='Prueba', linewidth=2, color='#FF5722')
    axes[1].set_xlabel('Épocas', fontsize=12)
    axes[1].set_ylabel('Precisión', fontsize=12)
    axes[1].set_title('Evolución de la Precisión', fontsize=14)
    axes[1].legend(fontsize=10)
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

# Graficar el historial de entrenamiento
plot_training_history(history)

La gráfica de pérdida debería mostrar una curva decreciente en ambas métricas, con la pérdida de prueba generalmente un poco más alta que la de entrenamiento. Si las dos curvas están muy separadas, indicando que la pérdida de prueba es significativamente mayor que la de entrenamiento, podríamos estar ante un caso de sobreajuste. Si ambas curvas se estabilizan en valores altos, el modelo está sousajustado y necesita más capacidad o más entrenamiento.

La gráfica de precisión muestra el complementario de la pérdida: a medida que la pérdida disminuye, la precisión debería aumentar. Una precisión del 100% en entrenamiento pero significativamente menor en prueba indicaría sobreajuste; precisiones bajas en ambos conjuntos indicarían que la red no ha aprendido los patrones fundamentales de los datos.

### Analizando el comportamiento del entrenamiento

La forma de las curvas de aprendizaje revela información importante sobre el proceso de optimización. En las primeras épocas, generalmente vemos una disminución rápida de la pérdida: la red aprende los patrones más obvios rápidamente. Luego viene una fase de aprendizaje más lento donde la red refina sus predicciones. Finalmente, la curva se aplana cuando la red ha encontrado un mínimo local de la función de pérdida.

El learning rate que usamos (0.5 en nuestro ejemplo) influye significativamente en esta dinámica. Un learning rate demasiado alto causa oscilaciones grandes en las curvas: la pérdida disminuye drásticamente pero luego sube, nunca convergiendo平稳mente. Un learning rate demasiado bajo hace que la convergencia sea extremadamente lenta, requiriendo muchas más épocas para alcanzar una precisión aceptable. El valor de 0.5 funciona bien para este problema relativamente simple, pero problemas más complejos requerirían experimentación más cuidadosa.

### Matriz de confusión

La precisión general oculta información importante sobre qué tipos de errores comete la red. Una flor setosa clasificada como virginica es un error diferente a una virginica clasificada como versicolor. La matriz de confusión desagrega los errores por clase, revelando dónde tiene dificultades el modelo.

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

def plot_confusion_matrix(y_true, y_pred, class_names):
    """
    Grafica la matriz de confusión normalizada.
    """
    cm = confusion_matrix(y_true, y_pred)
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm_normalized, annot=True, fmt='.2%', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicción', fontsize=12)
    plt.ylabel('Etiqueta Real', fontsize=12)
    plt.title('Matriz de Confusión', fontsize=14)
    plt.show()

    # Mostrar números absolutos también
    print("\nConteo de predicciones:")
    print(f"  Setosa:    {cm[0, 0]} correctos, {cm[0, 1]+cm[0, 2]} errores")
    print(f"  Versicolor: {cm[1, 1]} correctos, {cm[1, 0]+cm[1, 2]} errores")
    print(f"  Virginica:  {cm[2, 2]} correctos, {cm[2, 0]+cm[2, 1]} errores")

# Obtener predicciones finales
y_train_pred = predict(X_train, trained_params)
y_test_pred = predict(X_test, trained_params)

# Convertir one-hot a etiquetas
y_train_labels = np.argmax(y_train, axis=1)
y_test_labels = np.argmax(y_test, axis=1)

print("MATRIZ DE CONFUSIÓN - CONJUNTO DE PRUEBA")
print("="*45)
plot_confusion_matrix(y_test_labels, y_test_pred, iris.target_names)

Para el dataset Iris, esperamos que la red clasifique perfectamente las flores setosa (son linealmente separables de las otras dos) pero tenga algunas dificultades con versicolor y virginica, que tienen más solapamiento en el espacio de características. La matriz de confusión confirmará o refutará esta intuición.

### Predicciones de ejemplo

Para desarrollar intuición sobre lo que la red realmente está haciendo, veamos las predicciones para algunas flores específicas del conjunto de prueba. Mostraremos las probabilidades outputeadas por la red junto con la etiqueta real, lo que nos permitirá ver no solo si la predicción fue correcta, sino cuán confiada estaba la red en cada caso.

In [ ]:
def show_sample_predictions(X, y_true, y_pred, params, n_samples=6):
    """
    Muestra predicciones de ejemplo con probabilidades.
    """
    probabilities, _ = forward_propagation(X, params)

    # Seleccionar muestras aleatorias
    np.random.seed(123)
    indices = np.random.choice(len(X), n_samples, replace=False)

    print("EJEMPLOS DE PREDICCIONES")
    print("="*65)

    for i, idx in enumerate(indices):
        probs = probabilities[idx]
        pred_class = y_pred[idx]
        true_class = y_true[idx]

        print(f"\nEjemplo {i+1}:")
        print(f"  Especie real: {iris.target_names[true_class]}")
        print(f"  Predicción:   {iris.target_names[pred_class]}", end="")
        print(" ✓" if true_class == pred_class else " ✗")
        print(f"  Probabilidades:")
        for j, name in enumerate(iris.target_names):
            bar = "█" * int(probs[j] * 30)
            print(f"    {name:12s}: {probs[j]:.1%} {bar}")

# Mostrar predicciones de ejemplo
show_sample_predictions(X_test, y_test_labels, y_test_pred, trained_params)

Este análisis granular complementa la precisión agregada con comprensión cualitativa. Vemos casos donde la red está muy confiada (probabilidad superior al 90% para la clase correcta) y casos donde la decisión es más ajustada. Los casos donde la red se equivoca con alta confianza son particularmente interesantes: revelan patrones que el modelo ha aprendido incorrectamente o ambigüedades genuinas en los datos.

### Resultados finales

Al final del entrenamiento, resumimos los resultados en métricas claras que podemos comparar con nuestra implementación baseline y, más adelante, con Scikit-learn.

In [ ]:
def print_final_results(y_train_true, y_train_pred, y_test_true, y_test_pred):
    """
    Imprime métricas finales de rendimiento.
    """
    train_acc = np.mean(y_train_true == y_train_pred) * 100
    test_acc = np.mean(y_test_true == y_test_pred) * 100

    print("\n" + "="*55)
    print("RESULTADOS FINALES DE NUESTRA IMPLEMENTACIÓN")
    print("="*55)
    print(f"Precisión en entrenamiento: {train_acc:.2f}%")
    print(f"Precisión en prueba:        {test_acc:.2f}%")
    print("="*55)

print_final_results(y_train_labels, y_train_pred, y_test_labels, y_test_pred)

Nuestra implementación desde cero debería alcanzar una precisión de prueba entre el 85% y el 95%, dependiendo de la inicialización aleatoria y los hiperparámetros específicos. Esta variabilidad es normal en redes neuronales: diferentes inicializaciones llevan a diferentes mínimos locales de la función de pérdida.

Los resultados demuestran que nuestra implementación es funcional: la red aprende los patrones del dataset Iris y generaliza razonablemente a datos no vistos. Pero siempre hay margen de mejora. En la siguiente sección, compararemos nuestra implementación con MLPClassifier de Scikit-learn para ver qué gana uno cuando usa una biblioteca profesional optimizada.

# 10. Comparación con Scikit-learn

Hemos construido nuestra propia red neuronal desde cero, implementando forward propagation, backpropagation, y el bucle de entrenamiento manualmente. Funciona, pero ¿cómo se compara con una implementación profesional? Scikit-learn incluye MLPClassifier, un perceptrón multicapa optimizado que encapsula años de investigación y desarrollo. En esta sección entrenaremos un modelo equivalente con MLPClassifier, compararemos los resultados, y analizaremos qué ganan las bibliotecas profesionales y qué pierde el estudiante que las usa sin entender los fundamentos.

### MLPClassifier en acción

MLPClassifier de Scikit-learn es el equivalente funcional de nuestra implementación, pero con muchas más opciones y optimizaciones. Soporta múltiples capas ocultas de cualquier tamaño, diferentes funciones de activación, optimizadores avanzados como Adam, regularización automática, y muchos otros hiperparámetros que permiten afinar el comportamiento del modelo. Para hacer una comparación justa, configuraremos MLPClassifier con exactamente la misma arquitectura que nuestra red: dos capas ocultas de 16 y 8 neuronas, función de activación ReLU, y el optimizador SGD (Stochastic Gradient Descent) que es equivalente al Gradient Descent que implementamos.

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report

# Configurar MLPClassifier con arquitectura idéntica: 4→16→8→3
mlp = MLPClassifier(
    hidden_layer_sizes=(16, 8),  # Dos capas ocultas: 16 y 8 neuronas
    activation='relu',            # Misma función de activación
    solver='sgd',                 # Stochastic Gradient Descent
    learning_rate_init=0.5,       # Mismo learning rate
    max_iter=2000,                # Mismo número de épocas
    random_state=42,              # Para reproducibilidad
    early_stopping=False,         # Desactivar para comparación justa
    validation_fraction=0.0,      # Sin conjunto de validación
    verbose=False                 # Sin salida durante entrenamiento
)

# Entrenar el modelo
print("Entrenando MLPClassifier de Scikit-learn...")
mlp.fit(X_train, y_train_labels)
print("Entrenamiento completado.\n")

# Predicciones
y_train_pred_sklearn = mlp.predict(X_train)
y_test_pred_sklearn = mlp.predict(X_test)

# Precisiones
train_acc_sklearn = accuracy_score(y_train_labels, y_train_pred_sklearn) * 100
test_acc_sklearn = accuracy_score(y_test_labels, y_test_pred_sklearn) * 100

print("RESULTADOS DE SCIKIT-LEARN")
print("="*45)
print(f"Precisión en entrenamiento: {train_acc_sklearn:.2f}%")
print(f"Precisión en prueba:        {test_acc_sklearn:.2f}%")

### Comparación lado a lado

Ahora que ambos modelos están entrenados, podemos compararlos directamente. Nuestra implementación logró cierta precisión, y MLPClassifier probablemente logrará algo similar o ligeramente mejor. Lo interesante no es tanto la diferencia numérica (que puede variar entre ejecuciones debido a la inicialización aleatoria), sino los patrones de comportamiento y las diferencias en el proceso.

In [ ]:
# Comparación detallada
print("\n" + "="*65)
print("COMPARACIÓN: NUESTRA IMPLEMENTACIÓN vs SCIKIT-LEARN")
print("="*65)
print(f"{'Métrica':<25} {'Nuestra Red':>15} {'Scikit-learn':>15}")
print("-"*65)
print(f"{'Precisión entrenamiento':<25} {final_train_acc*100:>14.2f}% {train_acc_sklearn:>14.2f}%")
print(f"{'Precisión prueba':<25} {final_test_acc*100:>14.2f}% {test_acc_sklearn:>14.2f}%")
print("="*65)

La precisión de ambos modelos debería ser comparable, típicamente en el rango del 85% al 95% en el conjunto de prueba para el dataset Iris. Nuestra implementación desde cero funciona correctamente, lo cual valida que entendimos bien los fundamentos y los implementamos correctamente.

### ¿Dónde gana Scikit-learn?

Aunque las precisiones sean similares, MLPClassifier tiene ventajas significativas en otros aspectos. La primera es la velocidad: Scikit-learn está implementado en Cython y optimizado a bajo nivel, por lo que entrena significativamente más rápido que nuestra implementación en NumPy puro. Esta diferencia de velocidad se magnifica en datasets más grandes y arquitecturas más profundas.

La segunda ventaja es la robustez. MLPClassifier incluye técnicas que nosotros tuvimos que implementar manualmente o ni siquiera consideramos. El optimizador Adam ajusta automáticamente el learning rate para cada parámetro, converge más rápido y es menos sensible a la elección del learning rate inicial. La regularización L2 previene el sobreajuste penalizando pesos grandes. El early stopping detiene el entrenamiento automáticamente cuando detecta que la precisión en validación deja de mejorar, evitando desperdiciar épocas en un modelo que no mejora.

La tercera ventaja es la flexibilidad. MLPClassifier soporta diferentes funciones de activación (tanh, logistic, ReLU), diferentes optimizadores (SGD, Adam, L-BFGS), inicialización adaptativa, y muchos otros hiperparámetros que permiten adaptar el modelo a problemas muy diferentes. Nuestra implementación solo soporta ReLU con SGD básico.

### Análisis de predicciones diferentes

Cuando comparamos las predicciones de ambos modelos,会发现 que no siempre coinciden. Hay flores donde nuestra red dice setosa y MLPClassifier dice virginica, o viceversa. Examinar estos casos revela información interesante sobre el comportamiento de cada modelo.

In [ ]:
# Encontrar diferencias en predicciones
differences = (y_test_pred != y_test_pred_sklearn)
num_differences = np.sum(differences)

print(f"\nDiferencias en predicciones: {num_differences}/{len(y_test_pred)} flores")
print("="*65)

if num_differences > 0:
    print("Flores donde los modelos discrepan:\n")

    diff_indices = np.where(differences)[0]

    for idx in diff_indices[:5]:  # Mostrar hasta 5 ejemplos
        true_class = iris.target_names[y_test_labels[idx]]
        our_pred = iris.target_names[y_test_pred[idx]]
        sklearn_pred = iris.target_names[y_test_pred_sklearn[idx]]

        print(f"Flor {idx}:")
        print(f"  Especie real:        {true_class}")
        print(f"  Nuestra predicción:  {our_pred}")
        print(f"  Scikit-learn:        {sklearn_pred}")
        print()

Las discrepancias generalmente ocurren en flores que están cerca de la frontera entre clases, especialmente entre versicolor y virginica donde hay más solapamiento. En estos casos, pequeñas diferencias en los pesos aprendido pueden llevar a decisiones diferentes. Lo importante es que ambos modelos están capturando los mismos patrones generales, solo difieren en los detalles de las flores más ambiguas.

### ¿Cuál usar en la práctica?

La respuesta honesta es: depende del contexto. Para aprendizaje y experimentación, implementar desde cero tiene un valor educativo que ninguna biblioteca puede reemplazar. Cuando implementas forward y backward propagation manualmente, desarrollas una comprensión profunda de qué está sucediendo internamente. Cuando algo sale mal, puedes diagnosticar el problema porque sabes exactamente qué operación está fallando.

Para proyectos de producción, las bibliotecas profesionales son la elección correcta. Scikit-learn, TensorFlow o PyTorch ofrecen implementaciones optimizadas, probadas y mantenidas que manejan eficientemente datasets grandes, soportan hardware acelerado (GPUs), e incluyen técnicas avanzadas que tomarían semanas o meses implementar correctamente desde cero.

El practicante ideal conoce ambos mundos: entiende los fundamentos lo suficientemente bien para diseñar arquitecturas personalizadas, diagnosticar problemas de convergencia, e interpretar resultados, pero también sabe aprovechar las herramientas profesionales cuando corresponde. Este artículo te ha dado acceso al primer mundo; el segundo mundo está a solo un import MLPClassifier de distancia.

### La lección final

Lo que demuestra esta comparación es que los fundamentos importan más que la implementación específica. Nuestra red neuronal desde cero funciona porque implementamos correctamente los principios fundamentales: forward propagation, función de pérdida, backpropagation, y actualización de pesos. Scikit-learn funciona porque implementa los mismos principios con más optimización y características adicionales.

Cuando uses TensorFlow o PyTorch en el futuro, no estarás usando una caja negra misteriosa. Conocerás cada operación que ocurre detrás de las escenas. Cuando llames a loss.backward(), sabrás que está calculando exactamente los mismos gradientes que calculamos manualmente en la sección 7. Cuando llames a optimizer.step(), sabrá que está actualizando los pesos en la dirección negativa del gradiente, exactamente como hicimos con la fórmula simple W = W - learning_rate * gradient.

Este conocimiento es tu ventaja competitiva. Te permite debuggear cuando las cosas no funcionan, diseñar arquitecturas personalizadas para problemas específicos, y leer papers de investigación con comprensión real en lugar de memorización superficial. Las bibliotecas son herramientas; los fundamentos son la comprensión que te hace capaz de usar esas herramientas efectivamente.

# 11. Conclusión y siguientes pasos

A lo largo de este artículo hemos construido una red neuronal desde cero, implementando cada componente de manera explícita y comprendiendo las matemáticas que lo sustentan. Ha sido un viaje desde la neurona individual hasta una arquitectura de dos capas ocultas, pasando por forward propagation, backpropagation, y la comparación con herramientas profesionales. Es momento de consolidar lo aprendido y planificar los siguientes pasos en tu camino por el deep learning.

### Qué hemos aprendido

Los conceptos fundamentales que dominas ahora forman la base sobre la cual se construyen todas las arquitecturas de deep learning modernas, desde redes convolucionales para visión por computadora hasta transformers para procesamiento de lenguaje natural. La neurona artificial, en su simplicidad, es la unidad básica de computación: combina entradas con pesos, aplica una función de activación no lineal, y produce una salida. Cuando organizamos muchas neuronas en capas y conectamos cada capa con la siguiente, emerges la capacidad de representar funciones arbitrariamente complejas, un resultado conocido como teorema de aproximación universal.

Forward propagation es el proceso mediante el cual los datos fluyen desde la entrada hasta la predicción final, transformándose en cada etapa. Cada capa realiza una multiplicación matricial seguida de una función de activación no lineal, y la composición de múltiples capas permite representaciones jerárquicas cada vez más abstractas. La función de pérdida cuantifica el error de la predicción, y la entropía cruzada categórica es la elección natural para clasificación multiclase porque penaliza severamente las predicciones confiadas pero incorrectas.

Backpropagation es el corazón del aprendizaje. Usando la regla de la cadena del cálculo, distribuye el error hacia todos los pesos que contribuyeron a la predicción, calculando exactamente cuánto debe ajustarse cada parámetro para reducir la pérdida. Este cálculo, aunque algebraicamente tedioso, es conceptualmente simple: cada gradiente indica la dirección y magnitud del ajuste necesario. Gradient Descent usa estos gradientes para actualizar los pesos iterativamente, y suficientes iteraciones conducen a pesos que hacen predicciones precisas.

Nuestra implementación logró precisión competitiva con MLPClassifier de Scikit-learn, validando que los fundamentos están correctamente implementados. Pero más importante que los números es la comprensión desarrollada: cuando uses TensorFlow o PyTorch en el futuro, cada operación será familiar porque ya la has implementado tú mismo.

### Limitaciones de nuestra implementación

Nuestra red neuronal es funcional pero básica. Hay técnicas fundamentales que no incluimos porque el objetivo era la comprensión de los fundamentos, no la construcción de un modelo de producción. El optimizador Adam, que ajusta automáticamente el learning rate para cada parámetro, converge más rápido y es más robusto que el SGD básico que implementamos. La regularización L2 y el dropout previene el sobreajuste penalizando pesos grandes o apagando neuronas aleatoriamente durante el entrenamiento. El early stopping detiene automáticamente el entrenamiento cuando la precisión de validación deja de mejorar, evitando desperdiciar cómputo en un modelo que ya no mejora.

Nuestra arquitectura también es modesta: solo dos capas ocultas con 16 y 8 neuronas. Las redes modernas tienen cientos de capas con millones de neuronas, y requieren GPUs para entrenar en tiempo razonable. Pero los principios fundamentales siguen siendo los mismos; la escala y la optimización son las únicas diferencias significativas.

### Siguientes pasos en tu aprendizaje

El dataset Iris es un ejemplo pequeño y controlado. El siguiente paso natural es experimentar con problemas más desafiantes. El dataset MNIST de dígitos manuscritos tiene 60,000 imágenes de 28×28 píxeles, suficiente para explorar arquitecturas más profundas y técnicas de regularización. Fashion-MNIST ofrece la misma estructura pero con imágenes de ropa, presentando patrones más diversos. Estos datasets son suficientemente pequeños para entrenar en una laptop pero suficientemente grandes para requerir arquitecturas bien diseñadas.

Una vez que estés cómodo con problemas de clasificación de imágenes simples, el siguiente nivel son las redes convolucionales (CNNs). Las CNNs aplican filtros que detectan patrones locales —bordes, texturas, formas— y apilan estas características en representaciones cada vez más abstractas. Son la base de toda la visión por computadora moderna, desde reconocimiento facial hasta detección de tumores en imágenes médicas. PyTorch y TensorFlow ofrecen capas convolucionales listas para usar, y ahora que entiendes los fundamentos, puedes concentrarte en cómo diseñar arquitecturas efectivas en lugar de implementar las operaciones manualmente.

Para datos secuenciales —texto, series temporales, audio— las redes recurrentes (RNNs) y especialmente los transformers dominan el campo. Los transformers, la arquitectura detrás de GPT y otros modelos de lenguaje, usan un mecanismo llamado atención que permite a cada elemento de la secuencia «consultar» todos los demás para producir representaciones contextuales. Los conceptos que aprendiste aquí —capas, pesos, funciones de activación, backpropagation— se aplican directamente, solo que la arquitectura es más elaborada.

### Recursos recomendados

Para profundizar tu comprensión, varios recursos destacan por su claridad y rigor. La serie de videos de 3Blue1Brown sobre neural networks explica los conceptos con visualizaciones excepcionales que desarrollan intuición geométrica. El curso de Andrew Ng en Coursera cubre los fundamentos matemáticos con el rigor apropiado para practicantes. El libro «Deep Learning» de Goodfellow es la referencia teórica definitiva, aunque assume más matemáticas que lo que hemos visto aquí.

La práctica es insustituible. Implementa variaciones de tu red: prueba diferentes arquitecturas, observa cómo cambia el rendimiento. Implementa técnicas adicionales: regularización L2, dropout, diferentes optimizadores. Participa en competencias de Kaggle donde resolverás problemas reales con presión de tiempo y competencia. Cada proyecto completado consolida tu comprensión y revela nuevas preguntas que guiarán tu aprendizaje.

### Reflexión final

Las redes neuronales pueden parecer magia cuando las usas como caja negra, pero después de implementar una desde cero, la magia se transforma en matemáticas elegantes y principios claros. La neurona individual es simple; la composición de muchas neuronas en capas es poderosa. Forward propagation es determinista; backpropagation es el algoritmo que hace el aprendizaje posible. Los gradientes son la señal que guía el ajuste de pesos, y suficientes iteraciones de Gradient Descent producen pesos que capturan los patrones de los datos.

Este conocimiento es transferible a cualquier framework de deep learning. Cuando uses PyTorch y llames a loss.backward(), sabrás exactamente qué está calculando: la regla de la cadena aplicada a tu función de pérdida a través de cada capa de tu red. Cuando llames a optimizer.step(), sabrás que está actualizando cada peso en la dirección negativa de su gradiente. Esta comprensión te diferencia del practitioner que usa redes neuronales como herramienta misteriosa, y te permite diseñar, diagnosticar y optimizar modelos con confianza.

Has dado un paso significativo en tu viaje por el deep learning. Los fundamentos que dominas ahora son la base de sistemas que reconocen rostros, traducen idiomas, generan texto e imágenes, y impulsan los avances más emociantes en inteligencia artificial. Cada experto fue alguna vez un principiante que no se rindió. Continúa aprendiendo, experimentando y construyendo. Los modelos más sofisticados que construirás un día usan los mismos principios que implementaste hoy.

Gracias por acompañarme en este recorrido. Felicitaciones por completar el artículo.